In [1]:
from discreteUtil import *
import sys
import time
import pandas as pd

sys.path.insert(0, "binding")
import p3gasus_discrete_cpp as p3cpp

def testTimeCpp(method, ACTIONS, STARTS, base_type=p3cpp.BaseADGType.BASE_FORTED):
    ACTIONS = np.asarray(ACTIONS, dtype=np.int64)
    STARTS = np.asarray(STARTS, dtype=np.int64)
    start = time.time()
    if method is p3cpp.MAGE:
        exGraph = method(ACTIONS, STARTS, base_type)
    else:
        exGraph = method(ACTIONS, STARTS)
    end = time.time()
    edges = len(exGraph.edges()) - len(ACTIONS[0]) * (len(exGraph.robot_list()) - 1)
    return edges, end - start


In [2]:
actions, starts, free = oneTestCase(100, 50)

In [3]:
rows = []

python_methods = [
    ("OriginalADG", OriginalADG, {}),
    ("SAGE", SAGE, {}),
    ("FORTED", FORTED, {}),
    ("MAGE (FORTED)", MAGE, {"subMethod": FORTED}),
    ("MAGE (SAGE)", MAGE, {"subMethod": SAGE}),
]

cpp_methods = [
    ("OriginalADG", p3cpp.OriginalADG, {}),
    ("SAGE", p3cpp.SAGE, {}),
    ("FORTED", p3cpp.FORTED, {}),
    ("MAGE (FORTED)", p3cpp.MAGE, {"base_type": p3cpp.BaseADGType.BASE_FORTED}),
    ("MAGE (SAGE)", p3cpp.MAGE, {"base_type": p3cpp.BaseADGType.BASE_SAGE}),
]

for name, method, kwargs in python_methods:
    edges, seconds = testTime(method, actions, starts, **kwargs)
    rows.append({"Implementation": "Python", "Method": name, "Edges": edges, "Time (s)": seconds})

for name, method, kwargs in cpp_methods:
    edges, seconds = testTimeCpp(method, actions, starts, **kwargs)
    rows.append({"Implementation": "C++", "Method": name, "Edges": edges, "Time (s)": seconds})

results = pd.DataFrame(rows).sort_values(["Method", "Implementation"]).reset_index(drop=True)
results["Time (s)"] = results["Time (s)"].round(6)
results


,Implementation,Method,Edges,Time (s)
0,C++,FORTED,1048,0.001035
1,Python,FORTED,1048,0.014995
2,C++,MAGE (FORTED),938,0.012539
3,Python,MAGE (FORTED),938,0.038005
4,C++,MAGE (SAGE),938,0.013809
5,Python,MAGE (SAGE),938,0.051246
6,C++,OriginalADG,4479,0.040790
7,Python,OriginalADG,4479,13.038373
8,C++,SAGE,4479,0.001729
9,Python,SAGE,4479,0.064962


In [4]:
temp = FORTED(actions, starts)
temp.fileWrite("Debug/")

actions_cpp = np.asarray(actions, dtype=np.int64)
starts_cpp = np.asarray(starts, dtype=np.int64)
tempCpp = p3cpp.FORTED(actions_cpp, starts_cpp)
tempCpp.file_write("Debug/")
